# 2 - Big data (Datashader)

For large point sets the backend aggregates on the server with **Datashader**: `rasterize` (an aggregated,
recolourable raster with a live colorbar), `datashade` (a fully shaded RGB), `trajectory` (line density of
ordered tracks) and `hexbin` (hexagonal binning). `dynamic=False` bakes one deterministic frame; the default
`dynamic=True` re-shades per viewport (needs a live server).

**Setup** and data - 3000 Heidelberg bird observations, plus a few synthetic GPS tracks.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # deliberately silence all warnings to keep the rendered docs clean

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

birds = gpd.read_file(str(DATA / "heidelberg_birds_2023.geojson"))
birds["log_uncertainty"] = np.log10(birds["uncertainty_m"].clip(lower=1.0))

rng = np.random.default_rng(1)
t = np.linspace(0, 1, 400)
parts = []
for k, (x0, y0) in enumerate([(8.60, 49.35), (8.65, 49.40), (8.70, 49.45)]):
    parts.append(gpd.GeoDataFrame(
        {"track": np.full(len(t), k)},
        geometry=gpd.points_from_xy(x0 + 0.18 * t, y0 + 0.08 * np.sin(5 * t)),
        crs="EPSG:4326",
    ))
tracks = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs="EPSG:4326")
print("birds:", len(birds), "| tracks:", tracks["track"].nunique())

## `rasterize` - aggregate, recolour live

`rasterize` bins the points into a raster by a reduction (`count`, `mean`, ...) and keeps a live colorbar.

In [ ]:
m = InteractiveMap(crs=4326, title="bird observations -> rasterize (count)")
m.rasterize(birds, aggregator="count", dynamic=False, width=360, height=280)
show(m)

## `datashade` - shaded RGB

`datashade` colours server-side into an RGB image. A continuous column shades by magnitude.

In [ ]:
m = InteractiveMap(crs=4326, title="bird observations -> datashade (uncertainty)")
m.datashade(birds, column="log_uncertainty", cmap="viridis", dynamic=False, width=360, height=280)
show(m)

## A frozen colour scale: `clim` + `cnorm`

By default a datashaded frame re-autoranges. `clim=(vmin, vmax)` pins the span and `cnorm` sets the
normalisation, so a sequence of frames stays comparable.

In [ ]:
m = InteractiveMap(crs=4326, title="datashade with a frozen 0-4.5 scale")
m.datashade(birds, column="log_uncertainty", clim=(0.0, 4.5), cnorm="linear",
            cmap="plasma", dynamic=False, width=360, height=280)
show(m)

## `trajectory` - line density of tracks

`trajectory` datashades ordered track points as line density; `track_column` groups them into separate lines.

In [ ]:
m = InteractiveMap(crs=4326, title="three tracks (trajectory)")
m.trajectory(tracks, track_column="track", dynamic=False, width=360, height=280)
show(m)

## `hexbin` - hexagonal binning

`hexbin` bins points into hexagons - a quick density read with no Datashader server.

In [ ]:
m = InteractiveMap(crs=4326, title="bird density (hexbin)")
m.hexbin(birds, gridsize=28, aggregator="count")
show(m)